# Using PkgDiet MCP Server with LlamaIndex

This notebook demonstrates how to connect a LlamaIndex agent to the [PkgDiet](https://github.com/om-tajne/pkgdiet) MCP server.

PkgDiet is a local-first dependency guardrail that stops AI agents from hallucinating or installing bloated, insecure, or deprecated npm packages. By connecting it to LlamaIndex via the Model Context Protocol (MCP), your agent can evaluate npm packages for health, size, and deprecation status *before* writing code or recommending them.

In [ ]:
%pip install llama-index-core llama-index-llms-openai llama-index-tools-mcp

We will use the `llama-index-tools-mcp` integration to connect to the PkgDiet server over stdio.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."

from llama_index.llms.openai import OpenAI
from llama_index.core.agent import ReActAgent
from llama_index.tools.mcp import SimpleMCPToolSpec

# Initialize the PkgDiet MCP Server using npx
mcp_spec = SimpleMCPToolSpec(
    command="npx",
    args=["-y", "pkgdiet@2.0.1", "mcp"]
)

tools = mcp_spec.to_tool_list()

llm = OpenAI(model="gpt-4o")
agent = ReActAgent.from_tools(tools, llm=llm, verbose=True)

# Ask the agent to evaluate an npm package
response = agent.chat("Check if the npm package 'request' is safe to use in a new project. Use the check_dependency tool.")
print(str(response))